# RMT · Associative Retrieval Remember
논문 부록 C/E/I의 과제와 원본 RMT recurrence를 검증합니다.
T4 · D128/4층/메모리32 · effective batch512 · FP32.
`1 → 2 → 3 → 5쌍` curriculum, 단계별 생성 exact match ≥99%일 때 진행합니다.
최대3,000 update와 전체 학습 wall budget20분 중 먼저 도달한 조건에서 종료합니다.
최종 test 시간은 별도이며, 전체 200쌍 논문 재현은 아닙니다. GRT 비교는 RMT 결과 확인 후 진행합니다.


In [ ]:
from pathlib import Path
import sys, subprocess, torch
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR = Path('/content/GRT')
RUN_DIR = Path('/content/drive/MyDrive/GRT/runs/rmt_remember_t4_seed1234')
assert (REPO_DIR / 'configs/rmt_paper_remember.yaml').exists(), '수정본을 pull한 checkout이 필요합니다.'
assert torch.cuda.is_available(), 'T4 GPU runtime을 선택하세요.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
from grt.config import load_config
print(torch.cuda.get_device_name(0), 'VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
print('run:', RUN_DIR)


In [ ]:
# 동일 명령을 다시 실행하면 완료된 단계를 건너뛰고 해당 단계 last.pt에서 재개합니다.
subprocess.run([sys.executable, '-u', 'scripts/train.py',
                '--config', 'configs/rmt_paper_remember.yaml',
                '--output-dir', str(RUN_DIR), '--device', 'cuda'], cwd=REPO_DIR, check=True)


In [ ]:
import json
import matplotlib.pyplot as plt

summary = json.loads((RUN_DIR / 'summary.json').read_text())
evaluation = json.loads((RUN_DIR / 'evaluation.json').read_text())
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
offset = 0
for stage in summary['stages']:
    # Use the current RUN_DIR even if Drive mount paths changed.
    stage_dir = RUN_DIR / Path(stage['checkpoint']).parent.name
    rows = [json.loads(line) for line in (stage_dir / 'metrics.jsonl').read_text().splitlines()]
    rows = sorted({r['global_step']: r for r in rows if 'val/loss' in r}.values(), key=lambda r:r['global_step'])
    steps = [offset + r['global_step'] for r in rows]
    label = str(stage['num_pairs']) + ' pairs'
    axes[0].plot(steps, [r['val/loss'] for r in rows], label=label)
    axes[1].plot(steps, [r['val/autoregressive/exact_match'] for r in rows], label=label)
    offset += stage['global_step']
axes[0].set_ylabel('validation CE (value + EOS)')
axes[1].set_ylabel('generated exact match (value + EOS)')
axes[1].axhline(.99, color='gray', linestyle='--')
axes[1].set_ylim(0, 1.02)
for ax in axes:
    ax.set_xlabel('total optimizer updates')
    ax.legend()
    ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(RUN_DIR / 'convergence.png', dpi=150)
plt.show()
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(json.dumps(evaluation, indent=2, ensure_ascii=False))
print('공유: summary.json, evaluation.json, convergence.png 및 각 stage의 metrics.jsonl, resolved_config.yaml, metadata.json')
